In [9]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

Running on gpu39.storrs.hpc.uconn.edu. All is good!


In [10]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from scipy import stats

In [11]:
base = "/shared/healthinfolab/datasets/ABCD/Irritability/Clinical_Data/Irritability/Release_5.0/"

files = {
    0:  base + "abcd_cbcl_irr_index_release5.0_0m_long.csv",
    12: base + "abcd_cbcl_irr_index_release5.0_12m_long.csv",
    24: base + "abcd_cbcl_irr_index_release5.0_24m_long.csv",
    36: base + "abcd_cbcl_irr_index_release5.0_36m_long.csv",
    48: base + "abcd_cbcl_irr_index_release5.0_48m_long.csv",
}

In [12]:
dfs = []

for t, path in files.items():
    df = pd.read_csv(path)
    df = df[["src_subject_id", "cbcl_irr_index_cnst"]].copy()
    df["time"] = t
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

In [13]:
data = data.rename(columns={
    "src_subject_id": "id",
    "cbcl_irr_index_cnst": "irr"
})

data = data.dropna()

counts = data.groupby("id").size()
valid_ids = counts[counts >= 2].index
data = data[data["id"].isin(valid_ids)]

In [14]:
# Choose a small subset of subjects for testing
# FIX: set random seed for reproducibility
np.random.seed(42)
subset_size = 1000

subset_ids = np.random.choice(
    data["id"].unique(),
    size=subset_size,
    replace=False
)

data = data[data["id"].isin(subset_ids)].copy()

In [15]:
ids = data["id"].unique()
id_map = {v: i for i, v in enumerate(ids)}

data["id_i"] = data["id"].map(id_map)

# FIX: scale time to [0, 1] to improve sampler geometry.
# Raw values 0-48 combined with small slope estimates can cause
# poorly-scaled posterior geometry for NUTS.
time = data["time"].values / 48.0

y = data["irr"].values
person = data["id_i"].values

N = len(ids)

print(f"N subjects: {N}")
print(f"N observations: {len(y)}")
print(f"irr range: [{y.min():.2f}, {y.max():.2f}], mean: {y.mean():.2f}, std: {y.std():.2f}")

N subjects: 1000
N observations: 4220
irr range: [0.00, 6.00], mean: 0.83, std: 1.23


## Model Selection: Fit LCGA for K = 2 to 5

LCGA requires comparing models with different numbers of latent classes and selecting the best-fitting one. We fit K=2 through K=5 and compare using **WAIC** (Widely Applicable Information Criterion). We also report classification **entropy** per model as a measure of how cleanly subjects separate into classes.

**Key fixes from original version:**
- Replaced `chains=10` with `chains=4` (standard; more chains does not improve within-chain mixing)
- Increased tuning to 1000 steps for better NUTS adaptation in mixture models
- Priors now informed by the observed data scale
- Added model selection loop across K values
- Added convergence diagnostics (R-hat)
- Fixed class assignment (posterior mode instead of rounded mean)

In [ ]:
K_values = [2, 3, 4, 5]
results = {}

# scale time (important)
time_scaled = (time - time.mean()) / time.std()

for K in K_values:
    print(f"\n{'='*50}")
    print(f"Fitting LCGA with K = {K}")
    print(f"{'='*50}")

    with pm.Model() as model:

        # class probabilities
        pi = pm.Dirichlet("pi", a=np.ones(K))

        # subject class
        c = pm.Categorical("c", p=pi, shape=N)

        # break symmetry by giving slightly separated priors
        intercept = pm.Normal(
            "intercept",
            mu=np.linspace(y.mean()-y.std(), y.mean()+y.std(), K),
            sigma=y.std()/2,
            shape=K
        )

        slope = pm.Normal(
            "slope",
            mu=0,
            sigma=y.std()/4,
            shape=K
        )

        sigma = pm.HalfNormal("sigma", sigma=y.std()/2)

        mu = intercept[c[person]] + slope[c[person]] * time_scaled

        y_obs = pm.Normal(
            "y_obs",
            mu=mu,
            sigma=sigma,
            observed=y
        )

        trace = pm.sample(
            draws=1500,
            tune=1500,
            chains=4,
            target_accept=0.9,
            random_seed=42,
            idata_kwargs={"log_likelihood": True}
        )

        waic = az.waic(trace)

    results[K] = {"trace": trace, "model": model, "waic": waic}

    print(f"K={K} | WAIC: {waic.elpd_waic:.2f}")


Fitting LCGA with K = 2


Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [pi, intercept, slope, sigma]
>BinaryGibbsMetropolis: [c]


Output()

Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 291 seconds.
/home/rif17002/anaconda3/envs/honors/lib/python3.10/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)


TypeError: log likelihood not found in inference data object

In [ ]:
# --- Convergence diagnostics ---
# R-hat close to 1.0 = good mixing. Values > 1.01 are a warning.

for K in K_values:
    summary = az.summary(
        results[K]["trace"],
        var_names=["pi", "intercept", "slope", "sigma"]
    )
    max_rhat = summary["r_hat"].max()
    flag = "OK" if max_rhat < 1.01 else "WARNING: poor convergence"
    print(f"K={K} | max R-hat: {max_rhat:.4f}  [{flag}]")

In [ ]:
# --- WAIC model comparison table ---
# Higher (less negative) WAIC = better predictive fit

comparison_rows = []
for K in K_values:
    w = results[K]["waic"]
    comparison_rows.append({"K": K, "WAIC": round(w.elpd_waic, 2), "SE": round(w.se, 2)})

comparison_df = pd.DataFrame(comparison_rows).set_index("K")
best_K = comparison_df["WAIC"].idxmax()

print(comparison_df)
print(f"\nBest K by WAIC: K = {best_K}")

## Inspect the Best-Fitting Model

You can override `best_K` manually below if theoretical or parsimony considerations suggest a different solution.

In [ ]:
# Override here if desired, e.g.: best_K = 3
K     = best_K
trace = results[K]["trace"]
print(f"Using K = {K}")

In [ ]:
# --- Trace plots to visually inspect chain mixing ---
az.plot_trace(trace, var_names=["pi", "intercept", "slope", "sigma"])
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior summary for growth parameters ---
az.summary(trace, var_names=["pi", "intercept", "slope", "sigma"])

In [ ]:
# --- Plot class trajectories ---
posterior = trace.posterior

inter_post = posterior["intercept"].mean(("chain", "draw")).values  # (K,)
slope_post = posterior["slope"].mean(("chain", "draw")).values       # (K,)
pi_post    = posterior["pi"].mean(("chain", "draw")).values          # (K,)

t_scaled = np.linspace(0, 1, 100)
t_months = t_scaled * 48

fig, ax = plt.subplots(figsize=(8, 5))
for k in range(K):
    traj = inter_post[k] + slope_post[k] * t_scaled
    ax.plot(t_months, traj, label=f"Class {k+1} (π={pi_post[k]:.2f})")

ax.set_xlabel("Months")
ax.set_ylabel("Irritability")
ax.set_title(f"LCGA Trajectories (K={K})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Class assignment ---
#
# FIX: the original code used post.mean().round() on a categorical variable,
# which is methodologically wrong and sensitive to label switching.
#
# Correct approach:
#   - Posterior mode = most frequently sampled class (hard assignment)
#   - Posterior class probabilities = proportion of samples in each class (soft assignment)
#
# Both are reported below.

post = trace.posterior["c"].stack(sample=("chain", "draw")).values  # (N, n_samples)

n_samples = post.shape[1]

# Hard assignment: posterior mode
class_mode = stats.mode(post, axis=1).mode.flatten()

# Soft assignment: probability of belonging to each class
class_probs = np.stack(
    [(post == k).sum(axis=1) / n_samples for k in range(K)],
    axis=1
)  # (N, K)

# Assemble results dataframe
class_df = pd.DataFrame(
    class_probs,
    columns=[f"prob_class_{k+1}" for k in range(K)]
)
class_df.insert(0, "id", ids)
class_df["assigned_class"] = class_mode + 1  # 1-indexed

print("Class distribution:")
print(class_df["assigned_class"].value_counts().sort_index())
print()
class_df.head(10)

In [ ]:
# --- Classification entropy ---
# Measures uncertainty in class assignment.
# Normalized to [0, 1]: 0 = perfect separation, 1 = no separation.

eps = 1e-10
entropy_per_person = -np.sum(
    class_probs * np.log(class_probs + eps), axis=1
) / np.log(K)

avg_entropy = entropy_per_person.mean()
print(f"Average classification entropy: {avg_entropy:.4f}")
print(f"(0 = perfect separation, 1 = no separation)")

plt.figure(figsize=(6, 4))
plt.hist(entropy_per_person, bins=30, edgecolor="white")
plt.xlabel("Entropy")
plt.ylabel("Count")
plt.title(f"Classification Entropy (K={K}, mean={avg_entropy:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# --- Save class assignments ---
out_path = "lcga_class_assignments.csv"
class_df.to_csv(out_path, index=False)
print(f"Saved class assignments to {out_path}")